# 21 - ROS 2 Architecture

## What / Why / How

**What we are trying to do:** Learn ROS 2 concepts without needing ROS 2 installed: nodes, topics, services, actions, and dropped messages.

**Why this matters:** Modern robot projects are distributed systems. You need architecture instincts before adding hardware complexity.

**How we will do it:** Simulate publish/subscribe callbacks, a service call, and an action-style feedback/result flow in plain Python.

## Goal

Understand ROS 2 architecture without requiring ROS 2 installed.

You will model:

- Nodes
- Topics
- Services
- Actions
- QoS-like message dropping

In a new real system, use ROS 2 Lyrical LTS unless a specific robot, driver, or vendor stack still requires Jazzy, Kilted, or Humble.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
class Topic:
    def __init__(self, drop_probability=0.0):
        self.subscribers = []
        self.drop_probability = drop_probability

    def subscribe(self, callback):
        self.subscribers.append(callback)

    def publish(self, message):
        for callback in self.subscribers:
            if random.random() >= self.drop_probability:
                callback(message)

odom = Topic(drop_probability=0.05)
cmd_vel = Topic()
logs = []

def controller_callback(msg):
    error = np.array(msg["goal"]) - np.array(msg["pose"])
    command = 0.5 * error
    cmd_vel.publish({"vx": command[0], "vy": command[1]})

def motor_callback(msg):
    logs.append(msg)

odom.subscribe(controller_callback)
cmd_vel.subscribe(motor_callback)

for i in range(20):
    odom.publish({"pose": [i*0.05, 0.0], "goal": [1.0, 0.5]})

print("commands received:", len(logs))
print("last command:", logs[-1])

In [ ]:
def inverse_kinematics_service(request):
    x, y = request["target"]
    return {"joint_guess": [np.arctan2(y, x), 0.0], "success": True}

response = inverse_kinematics_service({"target": [1.0, 1.0]})
print(response)

In [ ]:
def navigate_action(goal, max_feedback=5):
    for step in range(max_feedback):
        yield {"type": "feedback", "progress": (step + 1) / max_feedback}
    yield {"type": "result", "success": True, "goal": goal}

for event in navigate_action([2.0, 3.0]):
    print(event)

## Exercises

1. Add a watchdog that stops motors when no command arrives.
2. Model a latched topic for robot description.
3. Explain when you would use a service versus an action.